# Services Data Cleaning & EDA – San Diego Youth Opportunity Project

**Author: Lauren Vo**

Built a clean, unified “services” dataset for San Diego County that we can use to map youth opportunity deserts.

**This notebook does the following:**

1. Loads raw service-related datasets from `data/raw/services`:
   - `ymca_sd.csv`
   - `sd_public_libraries.csv`
   - `Library.csv`
   - `Parks_SD.csv`
   - `rec_centers_datasd.csv`
   - `Recreation_Centers_SD.csv`
   - `Transit_Routes_GTFS.csv`
   - `Transit_Stops_GTFS_20251028.csv`
2. Cleans each dataset (column names, ZIP codes, coordinates, facility flags).
3. Merges overlapping sources (duplicate libraries, duplicate recreation centers).
4. Builds a unified **services master table** (YMCA, libraries, parks, rec centers).
5. Performs basic EDA (counts by provider / type, coordinate sanity checks, missingness).
6. Saves cleaned outputs to `data/interim/services` and `data/processed/services`.


In [28]:
from pathlib import Path
import numpy as np
import pandas as pd

candidates = [Path("."), Path(".."), Path("../..")]

PROJECT_ROOT = None
for cand in candidates:
    if (cand / "data" / "raw" / "services").exists():
        PROJECT_ROOT = cand.resolve()
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find data/raw/services from this notebook location.")

print("Using PROJECT_ROOT:", PROJECT_ROOT)

DATA_DIR = PROJECT_ROOT / "data"
RAW_SERVICES_DIR = DATA_DIR / "raw" / "services"
INTERIM_SERVICES_DIR = DATA_DIR / "interim" / "services"
PROCESSED_SERVICES_DIR = DATA_DIR / "processed" / "services"

INTERIM_SERVICES_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_SERVICES_DIR.mkdir(parents=True, exist_ok=True)

RAW_SERVICES_DIR, INTERIM_SERVICES_DIR, PROCESSED_SERVICES_DIR

Using PROJECT_ROOT: /Users/laurenvo/Documents/github/Mapping-Youth-Opportunity-Deserts


(PosixPath('/Users/laurenvo/Documents/github/Mapping-Youth-Opportunity-Deserts/data/raw/services'),
 PosixPath('/Users/laurenvo/Documents/github/Mapping-Youth-Opportunity-Deserts/data/interim/services'),
 PosixPath('/Users/laurenvo/Documents/github/Mapping-Youth-Opportunity-Deserts/data/processed/services'))

In [29]:
import re

def clean_zip(value):
    """Extract a 5-digit ZIP code as a string; return NaN if not found."""
    if pd.isna(value):
        return np.nan
    s = str(value)
    m = re.search(r"\d{5}", s)
    return m.group(0) if m else np.nan

def standardize_phone(value):
    """Simple phone cleaner: strip whitespace; leave formatting as is."""
    if pd.isna(value):
        return np.nan
    return str(value).strip()

def lower_strip(series):
    """Lowercase + strip string Series safely."""
    return series.astype(str).str.strip().str.lower()

def summarize_dataset(df, name):
    """Quick text summary for EDA."""
    print(f"\n=== {name} ===")
    print("Shape:", df.shape)
    print("Columns:", list(df.columns))
    print(df.head(3))
    print("\nMissing values (%):")
    print((df.isna().mean() * 100).round(1))


In [30]:
# File names (as they appear in data/raw/services)
paths = {
    "ymca": RAW_SERVICES_DIR / "ymca_sd.csv",
    "sd_public_libraries": RAW_SERVICES_DIR / "sd_public_libraries.csv",
    "library_all": RAW_SERVICES_DIR / "Library.csv",
    "parks": RAW_SERVICES_DIR / "Parks_SD.csv",
    "rec_centers_src1": RAW_SERVICES_DIR / "rec_centers_datasd.csv",
    "rec_centers_src2": RAW_SERVICES_DIR / "Recreation_Centers_SD.csv",
    "gtfs_routes": RAW_SERVICES_DIR / "Transit_Routes_GTFS.csv",
    "gtfs_stops": RAW_SERVICES_DIR / "Transit_Stops_GTFS_20251028.csv",
}

raw = {name: pd.read_csv(path, low_memory=False) for name, path in paths.items()}

for name, df in raw.items():
    summarize_dataset(df, name)



=== ymca ===
Shape: (23, 10)
Columns: ['org', 'name', 'address', 'city', 'state', 'zip', 'phone', 'fax', 'website', 'source']
                        org                      name                  address       city state    zip           phone             fax  \
0  YMCA of San Diego County   Border View Family YMCA          3601 Arey Drive  San Diego    CA  91154  (619) 428-9622  (619) 298-9262   
1  YMCA of San Diego County       Cameron Family YMCA    10123 Riverwalk Drive     Santee    CA  92071  (619) 449-9622  (619) 449-9624   
2  YMCA of San Diego County  Copley-Price Family YMCA  4300 El Cajon Boulevard  San Diego    CA  92105  (619) 280-9622  (619) 283-7586   

                                             website                             source  
0  https://www.ymcasd.org/locations/border-view-f...  https://www.ymcasd.org/locations/  
1  https://www.ymcasd.org/locations/cameron-famil...  https://www.ymcasd.org/locations/  
2  https://www.ymcasd.org/locations/copley-price-.

In [31]:
ymca = raw["ymca"].copy()

# Basic cleaning
ymca["zip"] = ymca["zip"].apply(clean_zip)
ymca["phone"] = ymca["phone"].apply(standardize_phone)
ymca["fax"] = ymca["fax"].apply(standardize_phone)

# Add required columns that don't exist yet
ymca["lat"] = np.nan
ymca["lon"] = np.nan
ymca["hours"] = np.nan
ymca["eligibility"] = np.nan
ymca["service_type"] = "ymca"

# Column order 
ymca = ymca[
    [
        "service_type",
        "org",
        "name",
        "address",
        "city",
        "state",
        "zip",
        "lat",
        "lon",
        "phone",
        "fax",
        "website",
        "hours",
        "eligibility",
        "source",
    ]
]

summarize_dataset(ymca, "YMCA (clean)")
ymca.to_csv(INTERIM_SERVICES_DIR / "ymca_clean.csv", index=False)



=== YMCA (clean) ===
Shape: (23, 15)
Columns: ['service_type', 'org', 'name', 'address', 'city', 'state', 'zip', 'lat', 'lon', 'phone', 'fax', 'website', 'hours', 'eligibility', 'source']
  service_type                       org                      name                  address       city state    zip  lat  lon  \
0         ymca  YMCA of San Diego County   Border View Family YMCA          3601 Arey Drive  San Diego    CA  91154  NaN  NaN   
1         ymca  YMCA of San Diego County       Cameron Family YMCA    10123 Riverwalk Drive     Santee    CA  92071  NaN  NaN   
2         ymca  YMCA of San Diego County  Copley-Price Family YMCA  4300 El Cajon Boulevard  San Diego    CA  92105  NaN  NaN   

            phone             fax                                            website  hours  eligibility  \
0  (619) 428-9622  (619) 298-9262  https://www.ymcasd.org/locations/border-view-f...    NaN          NaN   
1  (619) 449-9622  (619) 449-9624  https://www.ymcasd.org/locations/cameron-fa

In [32]:
sdpl = raw["sd_public_libraries"].copy()

sdpl["zip"] = sdpl["zip"].apply(clean_zip)
sdpl["phone"] = sdpl["phone"].apply(standardize_phone)

sdpl["fax"] = np.nan
sdpl["lat"] = np.nan
sdpl["lon"] = np.nan
sdpl["hours"] = np.nan
sdpl["eligibility"] = np.nan
sdpl["service_type"] = "library"

sdpl = sdpl[
    [
        "service_type",
        "org",
        "name",
        "address",
        "city",
        "state",
        "zip",
        "lat",
        "lon",
        "phone",
        "fax",
        "website",
        "hours",
        "eligibility",
        "source",
    ]
]

summarize_dataset(sdpl, "SD Public Libraries (clean)")
sdpl.to_csv(INTERIM_SERVICES_DIR / "sd_public_libraries_clean.csv", index=False)



=== SD Public Libraries (clean) ===
Shape: (38, 15)
Columns: ['service_type', 'org', 'name', 'address', 'city', 'state', 'zip', 'lat', 'lon', 'phone', 'fax', 'website', 'hours', 'eligibility', 'source']
  service_type                       org                             name                 address       city state    zip  lat  lon  \
0      library  San Diego Public Library                  Central Library           330 Park Blvd  San Diego    CA  92101  NaN  NaN   
1      library  San Diego Public Library  Allied Gardens/Benjamin Library           5188 Zion Ave  San Diego    CA  92120  NaN  NaN   
2      library  San Diego Public Library                   Balboa Library  4255 Mt. Abernathy Ave  San Diego    CA  92117  NaN  NaN   

            phone  fax                                            website  hours  eligibility  \
0  (619) 236-5800  NaN  https://www.sandiego.gov/public-library/centra...    NaN          NaN   
1  (619) 533-3970  NaN                                       

In [33]:
lib = raw["library_all"].copy()

# Infer org: for San Diego city libraries, label consistently
lib["org"] = np.where(
    lib["DISTRICT"].str.contains("San Diego Public", case=False, na=False),
    "San Diego Public Library",
    lib["DISTRICT"],
)

lib["name"] = lib["NAME"]
lib["address"] = lib["ADDRESS"]
lib["city"] = lib["CITY"].str.replace(", CA", "", regex=False).str.strip()
lib["state"] = "CA"
lib["zip"] = lib["ZIP"].apply(clean_zip)
lib["phone"] = lib["PHONE"].apply(standardize_phone)
lib["website"] = lib["WEBSITE"]
lib["source"] = lib["DATA_SRC"]

# Add missing fields
for col in ["fax", "lat", "lon", "hours", "eligibility"]:
    lib[col] = np.nan

lib["service_type"] = "library"

lib_clean = lib[
    [
        "service_type",
        "org",
        "name",
        "address",
        "city",
        "state",
        "zip",
        "lat",
        "lon",
        "phone",
        "fax",
        "website",
        "hours",
        "eligibility",
        "source",
    ]
]

summarize_dataset(lib_clean, "Library.csv (clean, before dedupe)")
lib_clean.to_csv(INTERIM_SERVICES_DIR / "library_multi_jur_clean.csv", index=False)

# Merge with SDPL & deduplicate
libraries_all = pd.concat([sdpl, lib_clean], ignore_index=True)

# Make a deduplication key: org + name + ZIP (normalized)
for col in ["org", "name", "zip"]:
    libraries_all[col] = libraries_all[col].astype(str).str.strip()

dedupe_key = (
    lower_strip(libraries_all["org"])
    + " | "
    + lower_strip(libraries_all["name"])
    + " | "
    + libraries_all["zip"].fillna("")
)

libraries_all["dedupe_key"] = dedupe_key

# Keep the first occurrence of each key
libraries_all = libraries_all.sort_values(["org", "name"]).drop_duplicates(
    subset=["dedupe_key"], keep="first"
)

libraries_all = libraries_all.drop(columns=["dedupe_key"])

summarize_dataset(libraries_all, "Libraries (merged + deduped)")

libraries_all.to_csv(PROCESSED_SERVICES_DIR / "libraries_all_clean.csv", index=False)



=== Library.csv (clean, before dedupe) ===
Shape: (92, 15)
Columns: ['service_type', 'org', 'name', 'address', 'city', 'state', 'zip', 'lat', 'lon', 'phone', 'fax', 'website', 'hours', 'eligibility', 'source']
  service_type          org                      name            address         city state    zip  lat  lon           phone  fax  \
0      library  Chula Vista       Civic Center Branch       365 F Street  Chula Vista    CA  91910  NaN  NaN  (619) 691-5069  NaN   
1      library  Chula Vista  South Chula Vista Branch  389 Orange Avenue  Chula Vista    CA  91911  NaN  NaN  (619) 585-5755  NaN   
2      library     Coronado           Coronado Public  640 Orange Avenue     Coronado    CA  92118  NaN  NaN  (619) 522-7390  NaN   

                                            website  hours  eligibility                    source  
0  https://www.chulavistaca.gov/departments/library    NaN          NaN               Chula Vista  
1  https://www.chulavistaca.gov/departments/library    N

In [34]:
lib = raw["library_all"].copy()

# Infer org: for San Diego city libraries, label consistently
lib["org"] = np.where(
    lib["DISTRICT"].str.contains("San Diego Public", case=False, na=False),
    "San Diego Public Library",
    lib["DISTRICT"],
)

lib["name"] = lib["NAME"]
lib["address"] = lib["ADDRESS"]
lib["city"] = lib["CITY"].str.replace(", CA", "", regex=False).str.strip()
lib["state"] = "CA"
lib["zip"] = lib["ZIP"].apply(clean_zip)
lib["phone"] = lib["PHONE"].apply(standardize_phone)
lib["website"] = lib["WEBSITE"]
lib["source"] = lib["DATA_SRC"]

# Add missing fields
for col in ["fax", "lat", "lon", "hours", "eligibility"]:
    lib[col] = np.nan

lib["service_type"] = "library"

lib_clean = lib[
    [
        "service_type",
        "org",
        "name",
        "address",
        "city",
        "state",
        "zip",
        "lat",
        "lon",
        "phone",
        "fax",
        "website",
        "hours",
        "eligibility",
        "source",
    ]
]

summarize_dataset(lib_clean, "Library.csv (clean, before dedupe)")
lib_clean.to_csv(INTERIM_SERVICES_DIR / "library_multi_jur_clean.csv", index=False)

# Merge with SDPL & deduplicate
libraries_all = pd.concat([sdpl, lib_clean], ignore_index=True)

# Make a deduplication key: org + name + ZIP (normalized)
for col in ["org", "name", "zip"]:
    libraries_all[col] = libraries_all[col].astype(str).str.strip()

dedupe_key = (
    lower_strip(libraries_all["org"])
    + " | "
    + lower_strip(libraries_all["name"])
    + " | "
    + libraries_all["zip"].fillna("")
)

libraries_all["dedupe_key"] = dedupe_key

# Keep the first occurrence of each key
libraries_all = libraries_all.sort_values(["org", "name"]).drop_duplicates(
    subset=["dedupe_key"], keep="first"
)

libraries_all = libraries_all.drop(columns=["dedupe_key"])

summarize_dataset(libraries_all, "Libraries (merged + deduped)")

libraries_all.to_csv(PROCESSED_SERVICES_DIR / "libraries_all_clean.csv", index=False)



=== Library.csv (clean, before dedupe) ===
Shape: (92, 15)
Columns: ['service_type', 'org', 'name', 'address', 'city', 'state', 'zip', 'lat', 'lon', 'phone', 'fax', 'website', 'hours', 'eligibility', 'source']
  service_type          org                      name            address         city state    zip  lat  lon           phone  fax  \
0      library  Chula Vista       Civic Center Branch       365 F Street  Chula Vista    CA  91910  NaN  NaN  (619) 691-5069  NaN   
1      library  Chula Vista  South Chula Vista Branch  389 Orange Avenue  Chula Vista    CA  91911  NaN  NaN  (619) 585-5755  NaN   
2      library     Coronado           Coronado Public  640 Orange Avenue     Coronado    CA  92118  NaN  NaN  (619) 522-7390  NaN   

                                            website  hours  eligibility                    source  
0  https://www.chulavistaca.gov/departments/library    NaN          NaN               Chula Vista  
1  https://www.chulavistaca.gov/departments/library    N

In [35]:
rec1 = raw["rec_centers_src1"].copy()
rec2 = raw["rec_centers_src2"].copy()

# Rename ID in rec1 so we can merge
rec1 = rec1.rename(columns={"fac_nm_id": "facility_name_id", "zip": "zip_src1"})
rec2 = rec2.rename(columns={"zipcode": "zip_src2"})

# Merge on facility ID + names (this matches the majority of rows)
rec_merged = rec1.merge(
    rec2,
    on=["facility_name_id", "rec_bldg", "park_name"],
    how="outer",
    suffixes=("_src1", "_src2"),
)

# Combine address
rec_merged["address"] = rec_merged["address_src2"].fillna(rec_merged["address_src1"])
rec_merged["city"] = rec_merged["city"].fillna("San Diego")  # default
rec_merged["zip"] = rec_merged["zip_src2"].fillna(rec_merged["zip_src1"])
rec_merged["zip"] = rec_merged["zip"].apply(clean_zip)

# Coordinates: lat/lng from src1 (WGS84), otherwise leave NaN (x,y are in a projected CRS)
rec_merged["lat"] = rec_merged["lat"]
rec_merged["lon"] = rec_merged["lng"]

# Basic org / service info
rec_merged["org"] = "City of San Diego Parks & Recreation"
rec_merged["service_type"] = "rec_center"
rec_merged["name"] = rec_merged["rec_bldg"]

# Square footage
rec_merged["sq_ft"] = rec_merged["sq_ft"].fillna(rec_merged["square_feet"])

# Facility flags – keep as-is but they already match your desired semantics pretty well
facility_cols = [
    "gymnasium",
    "teen_ctr",
    "game_room",
    "multi_purp",
    "multi_purpose",
    "adult_ctr",
    "comfort_st",
    "comfort_station",
    "dance_rm",
    "dance_room",
    "tinytot_rm",
    "tiny_tot_room",
    "weight_rm",
    "weight_room",
]

# Make sure they exist
facility_cols_present = [c for c in facility_cols if c in rec_merged.columns]

# Build a clean subset for analysis
rec_clean_cols = [
    "service_type",
    "org",
    "name",
    "rec_bldg",
    "park_name",
    "address",
    "city",
    "zip",
    "lat",
    "lon",
    "sq_ft",
    "year_built",
    "service_district",
    "community",
    "neighborhood",
    "cd",
]

# Some columns might not exist depending on the merge; guard for that
rec_clean_cols = [c for c in rec_clean_cols if c in rec_merged.columns]

rec_clean = rec_merged[rec_clean_cols + facility_cols_present].copy()
rec_clean["source"] = "rec_centers_datasd + Recreation_Centers_SD"

summarize_dataset(rec_clean, "Recreation Centers (merged)")

rec_clean.to_csv(PROCESSED_SERVICES_DIR / "rec_centers_all_clean.csv", index=False)



=== Recreation Centers (merged) ===
Shape: (67, 27)
Columns: ['service_type', 'org', 'name', 'rec_bldg', 'park_name', 'address', 'city', 'zip', 'lat', 'lon', 'sq_ft', 'service_district', 'community', 'neighborhood', 'teen_ctr', 'multi_purp', 'multi_purpose', 'adult_ctr', 'comfort_st', 'comfort_station', 'dance_rm', 'dance_room', 'tinytot_rm', 'tiny_tot_room', 'weight_rm', 'weight_room', 'source']
  service_type                                   org                  name              rec_bldg                   park_name  \
0   rec_center  City of San Diego Parks & Recreation  Hourglass Rec Center  Hourglass Rec Center  Mira Mesa Communty College   
1   rec_center  City of San Diego Parks & Recreation    Encanto Rec Center    Encanto Rec Center      Encanto Community Park   
2   rec_center  City of San Diego Parks & Recreation   Encanto Teen Center   Encanto Teen Center      Encanto Community Park   

                   address       city    zip        lat         lon    sq_ft  service_

In [36]:
parks = raw["parks"].copy()

parks["org"] = "City of San Diego Parks & Recreation"
parks["service_type"] = "park"
parks["name"] = parks["full_name"].fillna(parks["common_name"])

# address_lo looks like "5235 Maple St. , 92105"
parks["address"] = parks["address_lo"].str.replace(" ,", ",", regex=False).str.strip()

# Extract ZIP code from address_lo
parks["zip"] = parks["address_lo"].apply(clean_zip)

# Assume city is San Diego for these records
parks["city"] = "San Diego"
parks["state"] = "CA"

# Coordinates
parks["lon"] = parks["x_coord"]
parks["lat"] = parks["y_coord"]

# Facility flags we care about
park_facility_cols = [
    "playground",
    "tot_lot",
    "baseball_50_6",
    "baseball_90",
    "softball",
    "multi_purpose",
    "basketball",
    "tennis",
    "sand_vball",
    "comfort_station",
    "pickleball",
]

park_cols = [
    "service_type",
    "org",
    "name",
    "common_name",
    "full_name",
    "address",
    "city",
    "state",
    "zip",
    "lat",
    "lon",
    "acres",
    "use_restriction",
    "dedicated_park",
    "community",
    "service_district",
]

park_cols = [c for c in park_cols if c in parks.columns]
park_facility_cols_present = [c for c in park_facility_cols if c in parks.columns]

parks_clean = parks[park_cols + park_facility_cols_present].copy()
parks_clean["source"] = "Parks_SD"

summarize_dataset(parks_clean, "Parks (clean)")
parks_clean.to_csv(PROCESSED_SERVICES_DIR / "parks_all_clean.csv", index=False)



=== Parks (clean) ===
Shape: (301, 28)
Columns: ['service_type', 'org', 'name', 'common_name', 'full_name', 'address', 'city', 'state', 'zip', 'lat', 'lon', 'acres', 'use_restriction', 'dedicated_park', 'community', 'service_district', 'playground', 'tot_lot', 'baseball_50_6', 'baseball_90', 'softball', 'multi_purpose', 'basketball', 'tennis', 'sand_vball', 'comfort_station', 'pickleball', 'source']
  service_type                                   org                                           name                 common_name  \
0         park  City of San Diego Parks & Recreation                          Oak Neighborhood Park                 OAK PARK NP   
1         park  City of San Diego Parks & Recreation  San Carlos Community Park & Recreation Center  SAN CARLOS CP & REC CENTER   
2         park  City of San Diego Parks & Recreation                        Marcy Neighborhood Park                    MARCY NP   

                                       full_name                      a

In [37]:
routes = raw["gtfs_routes"].copy()
stops = raw["gtfs_stops"].copy()

# Minimal cleaning: column renaming for clarity
routes_clean = routes.rename(
    columns={
        "route_short_name": "route_short",
        "route_long_name": "route_long",
        "route_type_text": "route_type_text",
    }
).copy()

stops_clean = stops.rename(
    columns={
        "stop_id": "stop_id",
        "stop_name": "stop_name",
        "stop_lat": "lat",
        "stop_lon": "lon",
    }
).copy()

# Convert coordinates to numeric
stops_clean["lat"] = pd.to_numeric(stops_clean["lat"], errors="coerce")
stops_clean["lon"] = pd.to_numeric(stops_clean["lon"], errors="coerce")

summarize_dataset(routes_clean, "GTFS Routes (clean)")
summarize_dataset(stops_clean, "GTFS Stops (clean)")

routes_clean.to_csv(PROCESSED_SERVICES_DIR / "gtfs_routes_clean.csv", index=False)
stops_clean.to_csv(PROCESSED_SERVICES_DIR / "gtfs_stops_clean.csv", index=False)



=== GTFS Routes (clean) ===
Shape: (478, 16)
Columns: ['OID_', 'shape_id', 'route_id', 'route_short', 'route_long', 'route_type', 'agency_id', 'route_desc', 'route_url', 'route_color', 'route_text_color', 'route_type_text', 'RouteShapeName', 'route_color_RGB', 'route_text_color_RGB', 'Shape_Length']
   OID_ shape_id route_id route_short               route_long  route_type agency_id  route_desc                     route_url  \
0     1  3010111      301         101  Oceanside - UTC/VA/UCSD           3      NCTD         NaN  http://www.gonctd.com/breeze   
1     2  3010125      301         101  Oceanside - UTC/VA/UCSD           3      NCTD         NaN  http://www.gonctd.com/breeze   
2     3  3010130      301         101  Oceanside - UTC/VA/UCSD           3      NCTD         NaN  http://www.gonctd.com/breeze   

  route_color route_text_color route_type_text RouteShapeName route_color_RGB route_text_color_RGB   Shape_Length  
0      008C9B           FFFFFF             Bus    3010111_301

In [38]:
# All datasets we want in the "services" master
services_parts = []

services_parts.append(ymca)

services_parts.append(
    libraries_all.assign(
        service_type="library"  # ensure tag just in case
    )
)

# For parks & rec centers we need to map columns onto the common schema
def to_common_schema(df, service_type, org_col, name_col, address_col,
                     city_col, state_value="CA", zip_col="zip",
                     lat_col="lat", lon_col="lon",
                     phone_col=None, fax_col=None, website_col=None,
                     extra_cols=None, source_label=None):
    out = pd.DataFrame()
    out["service_type"] = service_type
    out["org"] = df[org_col]
    out["name"] = df[name_col]
    out["address"] = df[address_col]
    out["city"] = df[city_col]
    out["state"] = state_value
    out["zip"] = df[zip_col].apply(clean_zip) if zip_col in df.columns else np.nan
    out["lat"] = df[lat_col] if lat_col in df.columns else np.nan
    out["lon"] = df[lon_col] if lon_col in df.columns else np.nan

    out["phone"] = df[phone_col] if phone_col and phone_col in df.columns else np.nan
    out["fax"] = df[fax_col] if fax_col and fax_col in df.columns else np.nan
    out["website"] = df[website_col] if website_col and website_col in df.columns else np.nan

    out["hours"] = np.nan
    out["eligibility"] = np.nan

    if source_label is not None:
        out["source"] = source_label
    elif "source" in df.columns:
        out["source"] = df["source"]
    else:
        out["source"] = np.nan

    if extra_cols:
        for new_name, col in extra_cols.items():
            out[new_name] = df[col] if col in df.columns else np.nan

    return out

# Rec centers
rec_common = to_common_schema(
    rec_clean,
    service_type="rec_center",
    org_col="org",
    name_col="name",
    address_col="address",
    city_col="city",
    zip_col="zip",
    lat_col="lat",
    lon_col="lon",
    extra_cols={
        "park_name": "park_name",
        "sq_ft": "sq_ft",
    },
    source_label="rec_centers_all_clean",
)

# Parks
parks_common = to_common_schema(
    parks_clean,
    service_type="park",
    org_col="org",
    name_col="name",
    address_col="address",
    city_col="city",
    zip_col="zip",
    lat_col="lat",
    lon_col="lon",
    extra_cols={
        "acres": "acres",
        "community": "community",
        "service_district": "service_district",
    },
    source_label="parks_all_clean",
)

services_parts.extend([rec_common, parks_common])

services_master = pd.concat(services_parts, ignore_index=True)

# Final tidy: clean zips again, ensure numeric lat/lon
services_master["zip"] = services_master["zip"].apply(clean_zip)
services_master["lat"] = pd.to_numeric(services_master["lat"], errors="coerce")
services_master["lon"] = pd.to_numeric(services_master["lon"], errors="coerce")

summarize_dataset(services_master, "Services master (before final dedupe)")

# Remove exact-duplicate rows (if any)
services_master = services_master.drop_duplicates().reset_index(drop=True)

services_master.to_csv(PROCESSED_SERVICES_DIR / "services_master.csv", index=False)
services_master.head()



=== Services master (before final dedupe) ===
Shape: (521, 20)
Columns: ['service_type', 'org', 'name', 'address', 'city', 'state', 'zip', 'lat', 'lon', 'phone', 'fax', 'website', 'hours', 'eligibility', 'source', 'park_name', 'sq_ft', 'acres', 'community', 'service_district']
  service_type                       org                      name                  address       city state    zip  lat  lon  \
0         ymca  YMCA of San Diego County   Border View Family YMCA          3601 Arey Drive  San Diego    CA  91154  NaN  NaN   
1         ymca  YMCA of San Diego County       Cameron Family YMCA    10123 Riverwalk Drive     Santee    CA  92071  NaN  NaN   
2         ymca  YMCA of San Diego County  Copley-Price Family YMCA  4300 El Cajon Boulevard  San Diego    CA  92105  NaN  NaN   

            phone             fax                                            website  hours  eligibility  \
0  (619) 428-9622  (619) 298-9262  https://www.ymcasd.org/locations/border-view-f...    NaN     

,service_type,org,name,address,city,state,zip,lat,lon,phone,fax,website,hours,eligibility,source,park_name,sq_ft,acres,community,service_district
0,ymca,YMCA of San Diego County,Border View Family YMCA,3601 Arey Drive,San Diego,CA,91154,NaN,NaN,(619) 428-9622,(619) 298-9262,https://www.ymcasd.org/locations/border-view-f...,NaN,NaN,https://www.ymcasd.org/locations/,NaN,NaN,NaN,NaN,NaN
1,ymca,YMCA of San Diego County,Cameron Family YMCA,10123 Riverwalk Drive,Santee,CA,92071,NaN,NaN,(619) 449-9622,(619) 449-9624,https://www.ymcasd.org/locations/cameron-famil...,NaN,NaN,https://www.ymcasd.org/locations/,NaN,NaN,NaN,NaN,NaN
2,ymca,YMCA of San Diego County,Copley-Price Family YMCA,4300 El Cajon Boulevard,San Diego,CA,92105,NaN,NaN,(619) 280-9622,(619) 283-7586,https://www.ymcasd.org/locations/copley-price-...,NaN,NaN,https://www.ymcasd.org/locations/,NaN,NaN,NaN,NaN,NaN
3,ymca,YMCA of San Diego County,Dan McKinney Family YMCA,8355 Cliffridge Avenue,La Jolla,CA,92037,NaN,NaN,(858) 453-3483,(858) 452-3761,https://www.ymcasd.org/locations/dan-mckinney-...,NaN,NaN,https://www.ymcasd.org/locations/,NaN,NaN,NaN,NaN,NaN
4,ymca,YMCA of San Diego County,Escondido YMCA – A Campus for Community Well-B...,1050 N. Broadway,Escondido,CA,92026,NaN,NaN,(760) 745-7490,NaN,https://www.ymcasd.org/locations/escondido-ymc...,NaN,NaN,https://www.ymcasd.org/locations/,NaN,NaN,NaN,NaN,NaN


In [39]:
print("Total service sites:", len(services_master))

print("\nCounts by service_type:")
print(services_master["service_type"].value_counts())

print("\nTop providers (org):")
print(services_master["org"].value_counts().head(10))

print("\nMissing coordinates by service_type:")
print(
    services_master.assign(
        lat_missing=services_master["lat"].isna(),
        lon_missing=services_master["lon"].isna(),
    )
    .groupby("service_type")[["lat_missing", "lon_missing"]]
    .mean()
    .mul(100)
    .round(1)
)

print("\nLat/Lon ranges:")
print(services_master["lat"].describe())
print(services_master["lon"].describe())


Total service sites: 521

Counts by service_type:
service_type
library    130
ymca        23
Name: count, dtype: int64

Top providers (org):
org
City of San Diego Parks & Recreation    368
San Diego Public Library                 75
San Diego County                         38
YMCA of San Diego County                 23
Oceanside                                 8
Carlsbad                                  3
Chula Vista                               3
Coronado                                  1
Escondido                                 1
National City                             1
Name: count, dtype: int64

Missing coordinates by service_type:
              lat_missing  lon_missing
service_type                          
library             100.0        100.0
ymca                100.0        100.0

Lat/Lon ranges:
count    3.510000e+02
mean     5.330029e+03
std      9.924366e+04
min      3.246286e+01
25%      3.271990e+01
50%      3.279124e+01
75%      3.290313e+01
max      1.859362e+06
Na

In [40]:
# A compact table suitable for joining to tracts / exporting to GIS
services_for_mapping = services_master[
    [
        "service_type",
        "org",
        "name",
        "address",
        "city",
        "state",
        "zip",
        "lat",
        "lon",
        "source",
    ]
].copy()

services_for_mapping.to_csv(PROCESSED_SERVICES_DIR / "services_for_mapping.csv", index=False)
services_for_mapping.head()


,service_type,org,name,address,city,state,zip,lat,lon,source
0,ymca,YMCA of San Diego County,Border View Family YMCA,3601 Arey Drive,San Diego,CA,91154,NaN,NaN,https://www.ymcasd.org/locations/
1,ymca,YMCA of San Diego County,Cameron Family YMCA,10123 Riverwalk Drive,Santee,CA,92071,NaN,NaN,https://www.ymcasd.org/locations/
2,ymca,YMCA of San Diego County,Copley-Price Family YMCA,4300 El Cajon Boulevard,San Diego,CA,92105,NaN,NaN,https://www.ymcasd.org/locations/
3,ymca,YMCA of San Diego County,Dan McKinney Family YMCA,8355 Cliffridge Avenue,La Jolla,CA,92037,NaN,NaN,https://www.ymcasd.org/locations/
4,ymca,YMCA of San Diego County,Escondido YMCA – A Campus for Community Well-B...,1050 N. Broadway,Escondido,CA,92026,NaN,NaN,https://www.ymcasd.org/locations/


In [41]:
mask_parksrec = (
    services_master["service_type"].isna()
    & (services_master["org"] == "City of San Diego Parks & Recreation")
)

# building-ish → rec center
services_master.loc[mask_parksrec & services_master["sq_ft"].notna(), "service_type"] = "rec_center"

# land-only → park
services_master.loc[mask_parksrec & services_master["sq_ft"].isna(), "service_type"] = "park"


In [42]:
services_master["service_type"].value_counts(dropna=False)


service_type
park          306
library       130
rec_center     62
ymca           23
Name: count, dtype: int64

In [43]:
zip_num = pd.to_numeric(services_master["zip"], errors="coerce")

bad_zip = (zip_num < 91900) | (zip_num > 92299)
services_master.loc[bad_zip, "zip"] = np.nan


In [44]:
lat_ok = services_master["lat"].between(32, 33.5)
lon_ok = services_master["lon"].between(-117.7, -116)

services_master.loc[~(lat_ok & lon_ok), ["lat", "lon"]] = np.nan


In [45]:
subset_cols = ["service_type", "org", "name", "address", "city", "zip"]

services_master = (
    services_master
    .sort_values(["lat", "lon"], na_position="last")  # rows with lat/lon first
    .drop_duplicates(subset=subset_cols, keep="first")
    .reset_index(drop=True)
)


In [46]:
dup_mask = services_master.duplicated(subset=subset_cols, keep=False)
services_master[dup_mask].sort_values(subset_cols)


,service_type,org,name,address,city,state,zip,lat,lon,phone,fax,website,hours,eligibility,source,park_name,sq_ft,acres,community,service_district


In [47]:
# Quick check / standardization of service_type 
print("service_type value counts BEFORE standardization:")
print(services_master["service_type"].value_counts(dropna=False))

# Example mapping 
type_map = {
    "Library": "library",
    "library": "library",
    "YMCA": "ymca",
    "Rec Center": "rec_center",
    "Recreation Center": "rec_center",
    "Park": "park",
    "park": "park",
}

services_master["service_type"] = services_master["service_type"].replace(type_map)


service_type value counts BEFORE standardization:
service_type
park          306
library       130
rec_center     61
ymca           23
Name: count, dtype: int64


In [52]:
# Youth-serving flag (heuristic)
services_master["name_lower"] = services_master["name"].astype(str).str.lower()
services_master["org_lower"] = services_master["org"].astype(str).str.lower()

youth_keywords = [
    "youth", "teen", "teens", "after school", "afterschool",
    "boys & girls", "bgc", "club", "mentoring", "mentorship"
]
is_sdys = services_master["org_lower"].str.contains("san diego youth services", na=False)
services_master.loc[is_sdys, "youth_serving"] = True


services_master["youth_serving"] = False
for kw in youth_keywords:
    mask = (
        services_master["name_lower"].str.contains(kw, na=False)
        | services_master["org_lower"].str.contains(kw, na=False)
    )
    services_master.loc[mask, "youth_serving"] = True

# Drop helper cols
services_master = services_master.drop(columns=["name_lower", "org_lower"])

print("Youth-serving site count:", services_master["youth_serving"].sum())


Youth-serving site count: 4


In [49]:
print("Total sites:", len(services_master))

print("\nSites by service_type:")
print(services_master["service_type"].value_counts())

print("\nSites by org (top 10):")
print(services_master["org"].value_counts().head(10))

print("\nMissing coordinates by service_type (%):")
print(
    services_master.assign(
        lat_missing=services_master["lat"].isna(),
        lon_missing=services_master["lon"].isna(),
    )
    .groupby("service_type")[["lat_missing", "lon_missing"]]
    .mean()
    .mul(100)
    .round(1)
)

print("\nParks – acres summary:")
print(
    services_master.loc[services_master["service_type"]=="park", "acres"]
    .describe()
)

print("\nRec centers – sq_ft summary:")
print(
    services_master.loc[services_master["service_type"]=="rec_center", "sq_ft"]
    .describe()
)


Total sites: 520

Sites by service_type:
service_type
park          306
library       130
rec_center     61
ymca           23
Name: count, dtype: int64

Sites by org (top 10):
org
City of San Diego Parks & Recreation    367
San Diego Public Library                 75
San Diego County                         38
YMCA of San Diego County                 23
Oceanside                                 8
Carlsbad                                  3
Chula Vista                               3
Coronado                                  1
Escondido                                 1
National City                             1
Name: count, dtype: int64

Missing coordinates by service_type (%):
              lat_missing  lon_missing
service_type                          
library             100.0        100.0
park                  5.6          5.6
rec_center            1.6          1.6
ymca                100.0        100.0

Parks – acres summary:
count     301.000000
mean       29.607651
std       24

In [50]:
services_master.to_csv(PROCESSED_SERVICES_DIR / "services_master_cleaned.csv", index=False)
